# SPECTER Embedding Pipeline
**Capstone I · PMC Research Assistant**

- Input: cleaned DataFrame (44k PubMed articles)
- Output: `specter_embeddings.npy` + `specter_ids.npy`
- Platform: Kaggle T4 GPU (~30-40 min)

---
**Before running:**
1. Kaggle → Settings → Accelerator: **GPU T4 x2**
2. Upload your cleaned CSV as Kaggle Dataset (see Step 1)
3. Add HF_TOKEN as Kaggle Secret (see Step 2)

## Step 0 — Install Dependencies

In [ ]:
%%capture
!pip install sentence-transformers huggingface_hub

## Step 1 — Load HF Token from Kaggle Secrets

**Setup (einmalig):**
- Kaggle → dein Account → Settings → **Secrets** → Add Secret
- Name: `HF_TOKEN`, Value: dein Token von HuggingFace

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
print("Token loaded:", HF_TOKEN[:10], "...")

## Step 2 — Load Cleaned DataFrame

**Setup (einmalig):**
- Kaggle → Datasets → New Dataset → CSV hochladen
- Dann hier: Notebook → Add Data → dein Dataset
- Pfad anpassen falls nötig

In [ ]:
import pandas as pd
import numpy as np
import os

# Pfad anpassen — Kaggle Datasets landen unter /kaggle/input/
DATA_PATH = "/kaggle/input/pmc-cleaned/cleaned_articles.csv"  # <-- anpassen

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df)} articles")
print(df.columns.tolist())
df.head(2)

## Step 3 — Prepare Input Texts
SPECTER erwartet: `title [SEP] abstract`

In [ ]:
# Spaltennamen anpassen falls nötig
TITLE_COL    = "title"     # <-- anpassen
ABSTRACT_COL = "abstract"  # <-- anpassen
ID_COL       = "article_id"  # <-- anpassen (oder 'pmid', 'pmc_id' etc.)

# Fehlende Werte auffüllen
df[TITLE_COL]    = df[TITLE_COL].fillna("")
df[ABSTRACT_COL] = df[ABSTRACT_COL].fillna("")

# SPECTER Format
texts = (df[TITLE_COL] + " [SEP] " + df[ABSTRACT_COL]).tolist()
ids   = df[ID_COL].tolist()

print(f"Total texts: {len(texts)}")
print(f"Example: {texts[0][:120]}...")

## Step 4 — Load SPECTER Model

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# GPU Check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# SPECTER laden
model = SentenceTransformer(
    "allenai/specter",
    token=HF_TOKEN,
    device=device
)
print("Model loaded ✓")

## Step 5 — Generate Embeddings
~30-40 min auf T4 GPU für 44k Artikel

In [ ]:
import time

BATCH_SIZE = 64  # T4: 64, falls OOM → auf 32 reduzieren

print(f"Starting embedding generation...")
print(f"Articles: {len(texts)} | Batch size: {BATCH_SIZE}")
print(f"Estimated batches: {len(texts) // BATCH_SIZE + 1}")
print("-" * 50)

start = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,   # Fortschrittsbalken
    convert_to_numpy=True,
    normalize_embeddings=False  # Normalisierung später in FAISS-Schritt
)

elapsed = time.time() - start
print(f"\nDone in {elapsed/60:.1f} min")
print(f"Shape: {embeddings.shape}")  # sollte (44393, 768) sein

## Step 6 — Sanity Check

In [ ]:
# Shape check
assert embeddings.shape[0] == len(texts), "Mismatch: embeddings vs texts!"
assert embeddings.shape[1] == 768, "Expected 768 dimensions!"
assert len(ids) == len(texts), "Mismatch: ids vs texts!"

# NaN check
nan_count = np.isnan(embeddings).sum()
print(f"NaN values: {nan_count}")

# Quick similarity test — zwei ähnliche Paper sollten hohen Score haben
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(embeddings[0:1], embeddings[1:2])[0][0]
print(f"Sample cosine similarity (article 0 vs 1): {sim:.4f}")

print("\nAll checks passed ✓")

## Step 7 — Save Output Files

In [ ]:
OUTPUT_DIR = "/kaggle/working/"

# Embeddings speichern
emb_path = os.path.join(OUTPUT_DIR, "specter_embeddings.npy")
np.save(emb_path, embeddings)
print(f"Saved: specter_embeddings.npy ({embeddings.nbytes / 1e6:.1f} MB)")

# IDs speichern — KRITISCH für FAISS Mapping
ids_path = os.path.join(OUTPUT_DIR, "specter_ids.npy")
np.save(ids_path, np.array(ids))
print(f"Saved: specter_ids.npy")

# Verify
emb_check = np.load(emb_path)
ids_check  = np.load(ids_path, allow_pickle=True)
print(f"\nVerification:")
print(f"  embeddings shape: {emb_check.shape}")
print(f"  ids length:       {len(ids_check)}")
print(f"  first id:         {ids_check[0]}")
print("\nAll files saved ✓")
print("\nDownload from: Kaggle → Output → specter_embeddings.npy")

---
## Next Steps
1. **Download** `specter_embeddings.npy` + `specter_ids.npy` from Kaggle Output
2. **FAISS Index bauen** → `faiss_index.ipynb`
3. **RAG Pipeline** → FAISS + Groq LLM + Streamlit